In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
# import sys
# import os
# sys.path.append(os.path.expanduser("~/git/argus_pipe_filter"))
# # sys.path.append("")

# 機体点群の間引き
+ SCX900向けに衝突判定に用いられる機体点群を除去する
+ lightning_v3を作るためのプログラム一覧

## ライブラリ

In [ ]:
import os

import numpy as np
import pandas as pd

## 定数

In [ ]:
# notebookの実行パスからプロジェクトのルートへの相対パス
path_to_root = "../"

In [ ]:
from configparser import ConfigParser, ExtendedInterpolation

from argus_synchro import shared_app_config
from argus_synchro.config.app_config import AppConfig
# import shared_app_config

In [ ]:
app_ini = ConfigParser(interpolation=ExtendedInterpolation())
app_ini.read(f"{path_to_root}/config/settings.ini", "UTF-8")
# 共有メモリに反映
app_config = AppConfig(app_ini)
# sac = shared_app_config.SharedAppConfig()
# app_config = sac.read()

## objファイルから生成された機体点群を読み込む

In [ ]:
import argus_synchro.SubScrutinizer as SubScrt
from argus_synchro.config.machine_collision import load_machine_info
from argus_synchro.experiments import py_machine_info_to_cpp

# import SubScrutinizer as SubScrt
# from config.machine_collision import load_machine_info
# from experiments import py_machine_info_to_cpp

### jsonファイルの指定

In [ ]:
machine_dir = f"{path_to_root}/config/crane3d/collision_detection/SCX900-3/weighted"
json_machine_info = f"{path_to_root}/config/crane3d/collision_detection/SCX900-3/weighted/col_machine_info.jsonc"

### 読み込み

In [ ]:
(l_machine_col_weighted, _,_ ) = SubScrt.create_machine_points(
    machine_dir,
    app_config.LiDARPosition,
    l_col_machine_conf = py_machine_info_to_cpp(load_machine_info(json_machine_info)),
)

# 機体の各部位で色分けて表示

In [ ]:
import k3d
import numpy as np
import matplotlib.pyplot as plt

from argus_synchro.experiments.debug_vis.viewer_3d import create_simple_k3d_points

In [ ]:
def colors_from_colormap_discrete(name: str, n: int):
    """matplotlib のカラーマップを n 等分サンプリングし、0xRRGGBB のリストを返す。"""
    cmap = plt.get_cmap(name)
    xs = np.linspace(0, 1, n)
    out = []
    for x in xs:
        r, g, b, _ = cmap(x)
        R, G, B = int(round(r*255)), int(round(g*255)), int(round(b*255))
        out.append((R << 16) | (G << 8) | B)
    return out

In [ ]:
palette = colors_from_colormap_discrete('tab10', len(l_machine_col_weighted))

plot = k3d.plot()
for parts, color in zip(l_machine_col_weighted, palette):
    plot += create_simple_k3d_points(parts.machine_pcd_points, color=color, point_size=0.02)

plot.display()

# V3のデータ間引き結果を格納する

In [ ]:
from argus_synchro.experiments.machine_selection.SCX900 import np_downsample
# from experiments.machine_selection.SCX900 import np_downsample

### 定数の設定

In [ ]:
export_dir = f"{path_to_root}/config/crane3d/collision_detection/SCX900-3/test_v3"
os.makedirs(export_dir, exist_ok=True)

random_state = 20250710

In [ ]:
# 選んだ点群を入れるためのリスト
chosen_points = []

### upper partの点を格納する
1. 削減したい点の目標をn_target_pointsに入れて、それになるように点群を減らしている
2. weightedのl_machine_colから今取り出したいMachineCollisionBaseを取り出す
3. 取り出したMachineCollisionBaseの機体点群をn_target_pointsになるようにselect_upper_points_v4で選ぶ
4. (x,y)座標が意図した部分にある点をchosen_indで選ぶ
5. x,y座標を反転させる
6. 反転させた点群を該当箇所に書き込む

In [ ]:
from argus_synchro.experiments.machine_selection.SCX900 import select_upper_points_v4
# from experiments.machine_selection.SCX900 import select_upper_points_v4

In [ ]:
base_name = "interpolated_SCX900_01_upper_part"
cell_size = (0.24, 0.24, 0.08)

In [ ]:
n_target_points = len(np.loadtxt(f'{path_to_root}/config/crane3d/collision_detection/SCX900-3/lightning/{base_name}.csv', delimiter=" "))
n_target_points

In [ ]:
target_machine_parts = next(filter(lambda elem: os.path.splitext(str(elem))[0] == base_name, l_machine_col_weighted))
upper_points = target_machine_parts.machine_pcd_points

In [ ]:
first_points, second_points = select_upper_points_v4(upper_points, 0, cell_size, frac=0.15)
first_points = np_downsample(first_points, frac=0.75, random_state=random_state)

chosen_ind = (np.abs(first_points[:,0]) < 1) & (np.abs(first_points[:,1]) < 1.9)
first_points = first_points[~chosen_ind]

_saved_points = np.vstack([first_points, second_points])
_saved_points[:,0] = -1 * _saved_points[:,0]
_saved_points[:,1] = -1 * _saved_points[:,1]

np.savetxt(f"{export_dir}/{base_name}.csv", _saved_points, delimiter=" ")
chosen_points.append(_saved_points)

## CWの点を格納する
1. 削減したい点の目標をn_target_pointsに入れて、それになるように点群を減らしている
2. weightedのl_machine_colから今取り出したいMachineCollisionBaseを取り出す
3. 取り出したMachineCollisionBaseの機体点群をselect_cw_points_v2で選ぶ
4. select_cw_points_on_linesでも同じように点を選ぶ
5. ダウンサンプルや閾値による点群除去を行う
6. カウンタウェイトの背後に位置する点群は除去する
7. x,y座標を反転させる
8. 反転させた点群を該当箇所に書き込む

In [ ]:
from argus_synchro.experiments.machine_selection.SCX900 import select_cw_points_v2, select_cw_points_on_lines
# from experiments.machine_selection.SCX900 import select_cw_points_v2, select_cw_points_on_lines

In [ ]:
base_name = "interpolated_SCX900_02_CW"
cell_size_side = (0.48, 0.48, 0.48)
cell_size_surface = (0.24, 0.24, 0.24)
under_z_th = 0.05
downsample_frac = 0.1

In [ ]:
n_target_points = len(np.loadtxt(f'{path_to_root}/config/crane3d/collision_detection/SCX900-3/lightning/{base_name}.csv', delimiter=" "))
n_target_points

In [ ]:
target_machine_parts = next(filter(lambda elem: os.path.splitext(str(elem))[0] == base_name, l_machine_col_weighted))
cw_points = target_machine_parts.machine_pcd_points

In [ ]:
first_points, second_points = select_cw_points_v2(cw_points, 0, cell_size_side=cell_size_side, cell_size_surface=cell_size_surface)

# quantile: 0, 25, 50, 75%の高さの位置を取り出しつつ、不要な点は除去
third_points = select_cw_points_on_lines(
    cw_points,
    exclusion_file_regex="{path_to_root}/config/crane3d/collision_detection/SCX900-3/intermed_data/cw_group*_exclusion.npy",
)

# 下面を間引く
under_points = third_points[third_points[:,2] < under_z_th]
under_points = np_downsample(under_points, frac=downsample_frac, random_state=random_state)
third_points = np.vstack([
    third_points[third_points[:,2] >= under_z_th],
    under_points
])

# カウンタウェイトの背後の点は消す
third_points = third_points[(third_points[:,0] >= 3.4) | (np.abs(third_points[:,1]) >= 1.5)]

# x,y座標を-1倍して書き込む
_saved_points = np.vstack([first_points, second_points, third_points])
_saved_points[:,0] = -1 * _saved_points[:,0]
_saved_points[:,1] = -1 * _saved_points[:,1]

np.savetxt(f"{export_dir}/{base_name}.csv", _saved_points, delimiter=" ")
chosen_points.append(_saved_points)

## senkai_chushinの点を格納する
1. 削減したい点の目標をn_target_pointsに入れて、それになるように点群を減らしている
2. weightedのl_machine_colから今取り出したいMachineCollisionBaseを取り出す
3. 取り出したMachineCollisionBaseの機体点群をselect_senkai_chushin_points_v2で選ぶ
4. x,y座標を反転させる
5. 反転させた点群を該当箇所に書き込む

In [ ]:
from argus_synchro.experiments.machine_selection.SCX900 import select_senkai_chushin_points_v2
# from experiments.machine_selection.SCX900 import select_senkai_chushin_points_v2

In [ ]:
base_name = "interpolated_SCX900_03_senkai_chushin"
cell_size = (0.1, 0.1, 0.035)

In [ ]:
n_target_points = len(np.loadtxt(f'{path_to_root}/config/crane3d/collision_detection/SCX900-3/lightning/{base_name}.csv', delimiter=" "))
n_target_points

In [ ]:
target_machine_parts = next(filter(lambda elem: os.path.splitext(str(elem))[0] == base_name, l_machine_col_weighted))
senkai_chushin_points = target_machine_parts.machine_pcd_points

In [ ]:
first_points, second_points = select_senkai_chushin_points_v2(senkai_chushin_points, 0, cell_size, frac=0.5)

# x,y座標を-1倍して書き込む
_saved_points = np.vstack([first_points, second_points])
_saved_points[:,0] = -1 * _saved_points[:,0]
_saved_points[:,1] = -1 * _saved_points[:,1]

np.savetxt(f"{export_dir}/{base_name}.csv", _saved_points, delimiter=" ")

chosen_points.append(_saved_points)

## front_right_L_ji_partの点を格納する
1. 削減したい点の目標をn_target_pointsに入れて、それになるように点群を減らしている
2. weightedのl_machine_colから今取り出したいMachineCollisionBaseを取り出す
3. 取り出したMachineCollisionBaseの機体点群をselect_front_right_L_ji_points_v2で選ぶ
4. x,y座標を反転させる
5. 反転させた点群を該当箇所に書き込む

In [ ]:
from argus_synchro.experiments.machine_selection.SCX900 import select_front_right_L_ji_points_v2
# from experiments.machine_selection.SCX900 import select_front_right_L_ji_points_v2

In [ ]:
base_name = "interpolated_SCX900_04_front_right_L_ji_part"
cell_size = (0.1, 0.1, 0.035)
diff_y_th = 0.15
diff_x_th = 0.1

In [ ]:
n_target_points = len(np.loadtxt(f'{path_to_root}/config/crane3d/collision_detection/SCX900-3/lightning/{base_name}.csv', delimiter=" "))
n_target_points

In [ ]:
target_machine_parts = next(filter(lambda elem: os.path.splitext(str(elem))[0] == base_name, l_machine_col_weighted))
front_right_L_ji_points = target_machine_parts.machine_pcd_points

In [ ]:
first_points, second_points = select_front_right_L_ji_points_v2(front_right_L_ji_points, 0, cell_size, diff_y_th, diff_x_th, frac=0.5)

# x,y座標を-1倍して書き込む
_saved_points = np.vstack([first_points, second_points])
_saved_points[:,0] = -1 * _saved_points[:,0]
_saved_points[:,1] = -1 * _saved_points[:,1]

np.savetxt(f"{export_dir}/{base_name}.csv", _saved_points, delimiter=" ")

chosen_points.append(_saved_points)

## front_right_L_ji_partの点を格納する
1. 削減したい点の目標をn_target_pointsに入れて、それになるように点群を減らしている
2. weightedのl_machine_colから今取り出したいMachineCollisionBaseを取り出す
3. 取り出したMachineCollisionBaseの機体点群をselect_front_left_L_ji_points_v2で選ぶ
4. x,y座標を反転させる
5. 反転させた点群を該当箇所に書き込む

In [ ]:
from argus_synchro.experiments.machine_selection.SCX900 import select_front_left_L_ji_points_v2
# from experiments.machine_selection.SCX900 import select_front_left_L_ji_points_v2

In [ ]:
base_name = "interpolated_SCX900_05_front_left_L_ji_part"
cell_size = (0.1, 0.1, 0.035)
diff_y_th = 0.15
diff_x_th = 0.1

In [ ]:
n_target_points = len(np.loadtxt(f'{path_to_root}/config/crane3d/collision_detection/SCX900-3/lightning/{base_name}.csv', delimiter=" "))
n_target_points

In [ ]:
target_machine_parts = next(filter(lambda elem: os.path.splitext(str(elem))[0] == base_name, l_machine_col_weighted))
front_left_L_ji_points = target_machine_parts.machine_pcd_points

In [ ]:
first_points, second_points = select_front_left_L_ji_points_v2(front_left_L_ji_points, 0, cell_size, diff_y_th, diff_x_th, frac=0.5)

# x,y座標を-1倍して書き込む
_saved_points = np.vstack([first_points, second_points])
_saved_points[:,0] = -1 * _saved_points[:,0]
_saved_points[:,1] = -1 * _saved_points[:,1]

np.savetxt(f"{export_dir}/{base_name}.csv", _saved_points, delimiter=" ")

chosen_points.append(_saved_points)

## back_right_L_ji_partの点を格納する
1. 削減したい点の目標をn_target_pointsに入れて、それになるように点群を減らしている
2. weightedのl_machine_colから今取り出したいMachineCollisionBaseを取り出す
3. 取り出したMachineCollisionBaseの機体点群をselect_back_right_L_ji_points_v2で選ぶ
4. x,y座標を反転させる
5. 反転させた点群を該当箇所に書き込む

In [ ]:
from argus_synchro.experiments.machine_selection.SCX900 import select_back_right_L_ji_points_v2
# from experiments.machine_selection.SCX900 import select_back_right_L_ji_points_v2

In [ ]:
base_name = "interpolated_SCX900_06_back_right_L_ji_part"
cell_size = (0.1, 0.1, 0.035)
diff_y_th = 0.15
diff_x_th = 0.1

In [ ]:
n_target_points = len(np.loadtxt(f'{path_to_root}/config/crane3d/collision_detection/SCX900-3/lightning/{base_name}.csv', delimiter=" "))
n_target_points

In [ ]:
target_machine_parts = next(filter(lambda elem: os.path.splitext(str(elem))[0] == base_name, l_machine_col_weighted))
back_right_L_ji_points = target_machine_parts.machine_pcd_points

In [ ]:
first_points, second_points = select_back_right_L_ji_points_v2(back_right_L_ji_points, 0, cell_size, diff_y_th, diff_x_th, frac=0.5)

# x,y座標を-1倍して書き込む
_saved_points = np.vstack([first_points, second_points])
_saved_points[:,0] = -1 * _saved_points[:,0]
_saved_points[:,1] = -1 * _saved_points[:,1]

np.savetxt(f"{export_dir}/{base_name}.csv", _saved_points, delimiter=" ")

chosen_points.append(_saved_points)

## back_left_L_ji_partの点を格納する
1. 削減したい点の目標をn_target_pointsに入れて、それになるように点群を減らしている
2. weightedのl_machine_colから今取り出したいMachineCollisionBaseを取り出す
3. 取り出したMachineCollisionBaseの機体点群をselect_back_left_L_ji_points_v2で選ぶ
4. x,y座標を反転させる
5. 反転させた点群を該当箇所に書き込む

In [ ]:
from argus_synchro.experiments.machine_selection.SCX900 import select_back_left_L_ji_points_v2
# from experiments.machine_selection.SCX900 import select_back_left_L_ji_points_v2

In [ ]:
base_name = "interpolated_SCX900_07_back_left_L_ji_part"
cell_size = (0.1, 0.1, 0.035)
diff_y_th = 0.15
diff_x_th = 0.1

In [ ]:
n_target_points = len(np.loadtxt(f'{path_to_root}/config/crane3d/collision_detection/SCX900-3/lightning/{base_name}.csv', delimiter=" "))
n_target_points

In [ ]:
target_machine_parts = next(filter(lambda elem: os.path.splitext(str(elem))[0] == base_name, l_machine_col_weighted))
back_left_L_ji_points = target_machine_parts.machine_pcd_points

In [ ]:
first_points, second_points = select_back_left_L_ji_points_v2(back_left_L_ji_points, 0, cell_size, diff_y_th, diff_x_th, frac=0.5)

# x,y座標を-1倍して書き込む
_saved_points = np.vstack([first_points, second_points])
_saved_points[:,0] = -1 * _saved_points[:,0]
_saved_points[:,1] = -1 * _saved_points[:,1]

np.savetxt(f"{export_dir}/{base_name}.csv", _saved_points, delimiter=" ")

chosen_points.append(_saved_points)

## crawler_rightの点を格納する
1. 削減したい点の目標をn_target_pointsに入れて、それになるように点群を減らしている
2. weightedのl_machine_colから今取り出したいMachineCollisionBaseを取り出す
3. 取り出したMachineCollisionBaseの機体点群をselect_crawler_right_points_v2で選ぶ
4. select_crawler_right_on_linesでも同じように点を選ぶ
5. x,y座標を反転させる
6. 反転させた点群を該当箇所に書き込む

In [ ]:
from argus_synchro.experiments.machine_selection.SCX900 import select_crawler_right_points_v2, select_crawler_right_on_lines
# from experiments.machine_selection.SCX900 import select_crawler_right_points_v2, select_crawler_right_on_lines

In [ ]:
base_name = "interpolated_SCX900_08_crawler_right"
cell_size = (0.24, 0.24, 0.08)
cell_size_over = (0.12, 0.12, 0.04)

In [ ]:
n_target_points = len(np.loadtxt(f'{path_to_root}/config/crane3d/collision_detection/SCX900-3/lightning/{base_name}.csv', delimiter=" "))
n_target_points

In [ ]:
target_machine_parts = next(filter(lambda elem: os.path.splitext(str(elem))[0] == base_name, l_machine_col_weighted))
crawler_right_points = target_machine_parts.machine_pcd_points

In [ ]:
first_points, second_points = select_crawler_right_points_v2(crawler_right_points, 0, cell_size=cell_size)
third_points = select_crawler_right_on_lines(
    crawler_right_points,
    exclusion_file_regex=f"{path_to_root}/config/crane3d/collision_detection/SCX900-3/intermed_data/crawler_right_*_exclusion.npy"
)

# x,y座標を-1倍して書き込む
_saved_points = np.vstack([first_points, second_points, third_points])
_saved_points[:,0] = -1 * _saved_points[:,0]
_saved_points[:,1] = -1 * _saved_points[:,1]

np.savetxt(f"{export_dir}/{base_name}.csv", _saved_points, delimiter=" ")

chosen_points.append(_saved_points)

## crawler_leftの点を格納する
1. 削減したい点の目標をn_target_pointsに入れて、それになるように点群を減らしている
2. weightedのl_machine_colから今取り出したいMachineCollisionBaseを取り出す
3. 取り出したMachineCollisionBaseの機体点群をselect_crawler_left_pointsで選ぶ
4. select_crawler_left_on_linesでも同じように点を選ぶ
5. x,y座標を反転させる
6. 反転させた点群を該当箇所に書き込む

In [ ]:
from argus_synchro.experiments.machine_selection.SCX900 import select_crawler_left_points, select_crawler_left_on_lines
# from experiments.machine_selection.SCX900 import select_crawler_left_points, select_crawler_left_on_lines

In [ ]:
base_name = "interpolated_SCX900_09_crawler_left"
cell_size = (0.24, 0.24, 0.08)
cell_size_over = (0.12, 0.12, 0.04)

In [ ]:
n_target_points = len(np.loadtxt(f'{path_to_root}/config/crane3d/collision_detection/SCX900-3/lightning/{base_name}.csv', delimiter=" "))
n_target_points

In [ ]:
target_machine_parts = next(filter(lambda elem: os.path.splitext(str(elem))[0] == base_name, l_machine_col_weighted))
crawler_left_points = target_machine_parts.machine_pcd_points

In [ ]:
first_points, second_points = select_crawler_left_points(crawler_left_points, 0, cell_size=cell_size)
third_points = select_crawler_left_on_lines(
    crawler_left_points,
    exclusion_file_regex="{path_to_root}/config/crane3d/collision_detection/SCX900-3/intermed_data/crawler_left_*_exclusion.npy",
)

# x,y座標を-1倍して書き込む
_saved_points = np.vstack([first_points, second_points, third_points])
_saved_points[:,0] = -1 * _saved_points[:,0]
_saved_points[:,1] = -1 * _saved_points[:,1]

np.savetxt(f"{export_dir}/{base_name}.csv", _saved_points, delimiter=" ")

chosen_points.append(_saved_points)

## 点数の比較

In [ ]:
np.vstack([
    parts.machine_pcd_points
    for parts in l_machine_col_weighted
]).shape

In [ ]:
np.vstack(chosen_points).shape